# RGB-Only Person-Detection Baseline

## First make RGB work. Then make thermal work. Then make fusion work.

This notebook establishes the control experiment for the UAV Search and Rescue project. It trains and evaluates one ordinary RGB-only YOLO detector on the visible-light portion of the WiSARD dataset. The purpose is not to maximize performance immediately. The purpose is to create a clean, reproducible reference point that later thermal and multimodal experiments can be compared against.

A **baseline** is a simple, clearly documented experiment. Think of it as a measuring ruler: if a later RGB+thermal model scores higher, we need to know whether the improvement came from thermal information, alignment, fusion, a different split, or simply a different training recipe.

This notebook is the RGB control condition for the planned ladder:
- E0: RGB-only baseline, this notebook
- E1: thermal-only baseline
- E2: RGB+thermal basic middle fusion
- E3: RGB+thermal with feature alignment
- E4: RGB+thermal with attention fusion
- E5: RGB+thermal with multi-scale fusion

The source dataset is read-only at `D:/Hackathons/miniproject5sem/WiSARD_Multi_Modal_Sample`. This notebook never moves, copies, renames, edits, or deletes source images or annotations. It saves only references, metrics, plots, and model-run outputs under the project repository.

## Before the experiment: what we already know

The completed audit found 264 RGB images in `210417_MtErie_Enterprise_VIS_0003`, with one YOLO annotation file beside each image. The images are 3840 x 2160 pixels. Only class ID 0 occurs locally, and the local files do not provide a class-name file, so this notebook reports it conservatively as `class 0` rather than inventing a class name.

The files are sequential frames from one scene/sequence. That matters because neighboring frames may show almost the same people and background. A random image-level split could put near-duplicates in both training and test data and make the detector appear better than it really generalizes. We therefore use a deterministic chronological split: the first approximately 70% of frames for training, the next approximately 15% for validation, and the final approximately 15% for test.

The geometric investigation also found that RGB and thermal images are not proven to share a pixel grid. That is why this notebook deliberately uses RGB alone and does not resize, fuse, or align thermal data.

In [ ]:
# These imports make the experiment inspectable rather than hiding it in a single command.
# pathlib gives us safe cross-platform paths; NumPy and Pandas support tables;
# OpenCV reads images for inspection; Matplotlib draws qualitative examples.
from pathlib import Path
import csv
import json
import os
import random
import re
import sys
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

SEED = 20260905
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('OpenCV:', cv2.__version__)

## Section 1 - Discover the RGB data at runtime

### BEFORE
We identify the visible/RGB directory, image files, annotation files, frame IDs, dimensions, and annotation validity. This verifies that the notebook is using the expected data instead of silently relying on a hard-coded file list.

### DURING
The code searches folder names for `VIS` or `RGB`, extracts the final numeric token from each filename, and reads each annotation as a YOLO row. `count.txt` is metadata and is explicitly excluded. A YOLO row has five fields: class ID, normalized center x, normalized center y, normalized width, and normalized height.

### AFTER
The summary table should show 264 RGB images, 264 annotation files, 3840 x 2160 dimensions, one observed class ID (0), 1,022 boxes, and four images with zero objects. These are checks, not assumptions.

### WHY THIS MATTERS
If the dataset inventory is wrong, training metrics have no reliable meaning. A reproducible baseline begins with a reproducible input inventory.

In [ ]:
# The dataset is external to the repository and must remain read-only.
DATASET_ROOT = Path('D:/Hackathons/miniproject5sem/WiSARD_Multi_Modal_Sample')
if not DATASET_ROOT.exists():
    # This fallback helps when the notebook is moved within the same workspace.
    DATASET_ROOT = Path.cwd().parent.parent / 'WiSARD_Multi_Modal_Sample'

PROJECT_ROOT = Path('D:/Hackathons/miniproject5sem/UAV-Search-and-Rescue')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

RGB_RESULTS = PROJECT_ROOT / 'results' / 'rgb'
RGB_FIGURES = RGB_RESULTS / 'visualizations'
RGB_RESULTS.mkdir(parents=True, exist_ok=True)
RGB_FIGURES.mkdir(parents=True, exist_ok=True)
assert DATASET_ROOT.exists(), f'RGB dataset not found: {DATASET_ROOT}'

FRAME_PATTERN = re.compile(r'_(\d+)$')
rgb_folder_candidates = [folder for folder in DATASET_ROOT.iterdir() if folder.is_dir() and ('VIS' in folder.name.upper() or 'RGB' in folder.name.upper())]
if len(rgb_folder_candidates) != 1:
    raise RuntimeError(f'Expected one RGB folder, found: {rgb_folder_candidates}')
RGB_FOLDER = rgb_folder_candidates[0]

def frame_id_from_path(path):
    # The final numeric filename token is the sequence frame identifier.
    match = FRAME_PATTERN.search(path.stem)
    return int(match.group(1)) if match else None

def read_yolo_file(path):
    # Empty files are valid: they mean that this image has zero annotated objects.
    # Invalid rows are returned separately so they cannot disappear silently.
    valid_rows = []
    invalid_rows = []
    for raw_line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        line = raw_line.strip()
        if not line:
            continue
        fields = line.split()
        try:
            if len(fields) != 5:
                raise ValueError('expected 5 fields')
            class_id = int(fields[0])
            values = [float(value) for value in fields[1:]]
            if class_id < 0 or not all(0.0 <= value <= 1.0 for value in values):
                raise ValueError('class or normalized value out of range')
            valid_rows.append((class_id, *values))
        except ValueError:
            invalid_rows.append(raw_line)
    return valid_rows, invalid_rows

rgb_images = {}
rgb_annotations = {}
other_files = []
for path in sorted(RGB_FOLDER.rglob('*')):
    if not path.is_file():
        continue
    frame_id = frame_id_from_path(path)
    if path.suffix.lower() in {'.jpeg', '.jpg', '.png'} and frame_id is not None:
        rgb_images[frame_id] = path
    elif path.suffix.lower() == '.txt' and path.name.lower() != 'count.txt' and frame_id is not None:
        rgb_annotations[frame_id] = path
    else:
        other_files.append(path)

annotation_data = {}
invalid_annotation_rows = {}
for frame_id, annotation_path in rgb_annotations.items():
    annotation_data[frame_id], invalid_annotation_rows[frame_id] = read_yolo_file(annotation_path)

image_dimensions = {}
unreadable_images = []
for frame_id, image_path in rgb_images.items():
    image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
    if image is None:
        unreadable_images.append(str(image_path))
    else:
        image_dimensions[frame_id] = (image.shape[1], image.shape[0])

object_counts = {frame_id: len(annotation_data.get(frame_id, [])) for frame_id in rgb_images}
class_counts = Counter(row[0] for rows in annotation_data.values() for row in rows)
box_count = sum(object_counts.values())
summary = pd.DataFrame({
    'statistic': ['RGB folder', 'RGB images', 'RGB annotations', 'image extensions', 'dimensions', 'class IDs', 'bounding boxes', 'zero-object images', 'malformed annotation files', 'unreadable images', 'metadata files ignored'],
    'value': [RGB_FOLDER.name, len(rgb_images), len(rgb_annotations), sorted({path.suffix.lower() for path in rgb_images.values()}), sorted(set(image_dimensions.values())), sorted(class_counts), box_count, sum(count == 0 for count in object_counts.values()), sum(bool(rows) for rows in invalid_annotation_rows.values()), len(unreadable_images), [path.name for path in other_files]],
})
display(summary)
assert len(rgb_images) == 264
assert len(rgb_annotations) == 264
assert box_count == 1022
assert not unreadable_images
assert all(not rows for rows in invalid_annotation_rows.values())
summary.to_csv(RGB_RESULTS / 'rgb_dataset_summary.csv', index=False)

### What happened?
The code discovered the RGB folder and checked every image and annotation file. The assertions intentionally stop the notebook if the known audit facts change.

### What does it mean?
The baseline is using the intended visible-light stream, not the thermal folder or the `count.txt` summaries. Class ID 0 is observed, but its semantic name is not available in the local files.

### Is the result reasonable?
Yes if the table reports the audited counts and zero malformed/unreadable files.

### What should we do next?
Inspect representative images and annotations visually before asking a detector to learn from them.

## Section 3 - Visualize RGB images and YOLO boxes

### BEFORE
We display a small, representative sample: an empty image if available, an image with one object, an image with multiple objects, and frames from different parts of the sequence.

### DURING
YOLO coordinates are normalized fractions. For an image of width `W` and height `H`, pixel center is `(center_x * W, center_y * H)`, pixel width is `box_width * W`, and pixel height is `box_height * H`. The rectangle's upper-left corner is the center minus half the width and height.

### AFTER
The plots show the source image and the same image with annotation rectangles. Tiny aerial persons will occupy relatively few pixels, which makes localization and classification difficult.

### WHY THIS MATTERS
A detector cannot learn a trustworthy target definition if the labels are misunderstood. Visualization also reveals whether small-object difficulty is likely to dominate later errors.

In [ ]:
def yolo_to_pixel_boxes(rows, image_shape):
    # This conversion uses the dimensions of this specific image.
    # We never assume that another modality or another resolution has the same coordinates.
    image_height, image_width = image_shape[:2]
    pixel_boxes = []
    for class_id, center_x, center_y, width, height in rows:
        pixel_width = width * image_width
        pixel_height = height * image_height
        pixel_center_x = center_x * image_width
        pixel_center_y = center_y * image_height
        pixel_boxes.append({
            'class_id': class_id,
            'left': pixel_center_x - pixel_width / 2,
            'top': pixel_center_y - pixel_height / 2,
            'width': pixel_width,
            'height': pixel_height,
            'center_x': pixel_center_x,
            'center_y': pixel_center_y,
        })
    return pixel_boxes

def show_rgb_with_boxes(frame_id, save_name):
    # Load only for reading and plotting. No pixels are written back to the source.
    image = cv2.imread(str(rgb_images[frame_id]), cv2.IMREAD_COLOR)
    boxes = yolo_to_pixel_boxes(annotation_data.get(frame_id, []), image.shape)
    figure, axis = plt.subplots(figsize=(12, 7))
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    for box in boxes:
        rectangle = plt.Rectangle((box['left'], box['top']), box['width'], box['height'], fill=False, edgecolor='lime', linewidth=2)
        axis.add_patch(rectangle)
        axis.text(box['left'], box['top'], f'class {box["class_id"]}', color='black', backgroundcolor='lime')
    axis.set_title(f'RGB frame {frame_id} | {image.shape[1]} x {image.shape[0]} | objects={len(boxes)}')
    axis.set_xlabel('pixel x coordinate'); axis.set_ylabel('pixel y coordinate')
    figure.tight_layout()
    figure.savefig(RGB_FIGURES / save_name, dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(figure)

# A simple numerical example makes the coordinate system concrete.
example_row = annotation_data[next(frame for frame, count in object_counts.items() if count > 0)][0]
print('Example YOLO row:', example_row)
print('For RGB 3840x2160, its pixel center is:', (example_row[1] * 3840, example_row[2] * 2160))

candidate_frames = [frame for frame, count in object_counts.items() if count == 0][:1]
candidate_frames += [frame for frame, count in object_counts.items() if count == 1][:1]
candidate_frames += [frame for frame, count in object_counts.items() if count >= 4][:2]
candidate_frames += [0, 66, 132, 198, 263]
selected_frames = list(dict.fromkeys(frame for frame in candidate_frames if frame in rgb_images))
for position, frame_id in enumerate(selected_frames):
    show_rgb_with_boxes(frame_id, f'rgb_example_{position:02d}_frame_{frame_id:04d}.png')

### What happened?
The notebook converted normalized coordinates into pixel rectangles and saved representative figures under `results/rgb/visualizations`.

### What does it mean?
The boxes describe the annotated targets in the RGB coordinate system. Their small pixel dimensions explain why aerial-person detection is a small-object problem.

### Is the result reasonable?
Boxes should stay inside or close to image boundaries and should appear over people. A misplaced rectangle would indicate a parsing or coordinate-conversion problem.

### What should we do next?
Quantify class and object-count distributions, then construct a split that respects the sequence.

## Section 4 - Class and object-count distribution

### BEFORE
We count classes, boxes, and objects per image. An image-level distribution answers a different question from a box-level distribution: one image may contain many people.

### DURING
The code creates tables and a histogram. It verifies whether class IDs other than 0 exist instead of assuming that the audit result will always remain unchanged.

### AFTER
We can see how often images are empty, single-object, or crowded.

### WHY THIS MATTERS
The distribution affects loss balance, qualitative sampling, and interpretation of precision/recall. The local files do not contain a class-name mapping, so we report class 0 without inventing semantics.

In [ ]:
class_distribution = pd.DataFrame({'class_id': list(class_counts.keys()), 'bounding_boxes': list(class_counts.values())})
objects_per_image = pd.DataFrame({'frame_id': list(object_counts.keys()), 'objects': list(object_counts.values())})
print('Class semantics are unavailable in the local files; class IDs are reported literally.')
display(class_distribution)
display(objects_per_image['objects'].value_counts().sort_index().rename('image_count').to_frame())

figure, axis = plt.subplots(figsize=(9, 4))
objects_per_image['objects'].plot.hist(bins=range(0, int(objects_per_image.objects.max()) + 2), align='left', rwidth=0.85, ax=axis, color='steelblue')
axis.set_title('RGB objects per image')
axis.set_xlabel('number of annotated objects'); axis.set_ylabel('number of images')
figure.tight_layout()
figure.savefig(RGB_FIGURES / 'rgb_object_count_distribution.png', dpi=130, bbox_inches='tight')
plt.show()
plt.close(figure)

### What happened?
The distribution was computed from annotation rows, not from `count.txt`.

### What does it mean?
The model will see a mixture of empty and crowded images. Only class ID 0 is locally observable, but its human-readable name is not supplied by the dataset files.

### Is the result reasonable?
It should agree with the audit: 1,022 boxes and four zero-object RGB images.

### What should we do next?
Create a chronological split and record it as references rather than making another copy of the data.

## Section 5 - Deterministic sequence-aware train/validation/test split

### BEFORE
The data is a continuous numbered sequence. We must avoid random image-level splitting because adjacent frames may contain nearly the same scene.

### DURING
Frames are sorted chronologically and divided into contiguous blocks of approximately 70%, 15%, and 15%. The split file stores absolute image and annotation paths as references. It does not copy images or labels.

### AFTER
The table reports the frame ranges, image counts, and object counts for each partition. The test set is kept untouched by training and validation decisions.

### WHY THIS MATTERS
This is a more honest test of later frames in the same sequence. It does not solve every generalization problem, but it makes temporal leakage visible and reproducible.

In [ ]:
ordered_frames = sorted(rgb_images)
train_end = int(len(ordered_frames) * 0.70)
val_end = train_end + int(len(ordered_frames) * 0.15)
split_frames = {
    'train': ordered_frames[:train_end],
    'val': ordered_frames[train_end:val_end],
    'test': ordered_frames[val_end:],
}

split_rows = []
for split_name, frame_ids in split_frames.items():
    for frame_id in frame_ids:
        split_rows.append({
            'split': split_name,
            'frame_id': frame_id,
            'image_path': str(rgb_images[frame_id].resolve()),
            'annotation_path': str(rgb_annotations.get(frame_id, '').resolve()) if frame_id in rgb_annotations else '',
            'object_count': object_counts[frame_id],
        })
split_table = pd.DataFrame(split_rows)
split_table.to_csv(RGB_RESULTS / 'rgb_sequence_split.csv', index=False)

split_summary = split_table.groupby('split').agg(
    images=('frame_id', 'count'),
    first_frame=('frame_id', 'min'),
    last_frame=('frame_id', 'max'),
    ground_truth_boxes=('object_count', 'sum'),
).reset_index()
display(split_summary)
print('Split policy: chronological contiguous blocks; no image files were copied.')

### What happened?
The notebook created a deterministic chronological reference file under `results/rgb/rgb_sequence_split.csv`.

### What does it mean?
Training sees earlier frames, validation sees the middle block, and test sees the later block. The exact frame boundaries are recorded, so later baselines can reproduce them.

### Is the result reasonable?
The blocks should be contiguous, non-overlapping, and together contain all 264 frames.

### What should we do next?
Explain the model input resolution and letterboxing before training.

## Section 6 - Preprocessing and model configuration

The sensor resolution is 3840 x 2160, but a detector usually processes a smaller model input. This is a computational choice, not a claim that the camera captured fewer pixels. Larger inputs preserve more tiny-person detail but require more memory and time. Smaller inputs are faster but can make tiny people disappear after resizing.

The model uses aspect-ratio-aware letterboxing: one scale factor resizes both axes and padding fills the remaining canvas. Direct stretching would change object shapes. Ultralytics handles this preprocessing during training and inference.

Because this environment reports no CUDA GPU, the reproducible default below is CPU-aware: `imgsz=640`, batch size 2, and a small smoke-training run. The notebook records this limitation rather than pretending it is a production training configuration. Set `RUN_FULL_TRAINING = True` on a suitable GPU when a longer baseline run is intended.

In [ ]:
# These settings define the baseline recipe and are saved with the results.
MODEL_NAME = 'yolo11n.pt'
INPUT_SIZE = 640
BATCH_SIZE = 2 if not torch.cuda.is_available() else 8
EPOCHS = 1
RUN_FULL_TRAINING = False
CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50
DEVICE = '0' if torch.cuda.is_available() else 'cpu'

configuration = {
    'model': MODEL_NAME, 'input_size': INPUT_SIZE, 'epochs': EPOCHS,
    'batch_size': BATCH_SIZE, 'seed': SEED, 'device': DEVICE,
    'pretrained': True, 'confidence_threshold': CONFIDENCE_THRESHOLD,
    'iou_threshold': IOU_THRESHOLD, 'split_policy': 'chronological 70/15/15',
    'source_dataset': str(DATASET_ROOT.resolve()),
}
(RGB_RESULTS / 'rgb_baseline_config.json').write_text(json.dumps(configuration, indent=2), encoding='utf-8')
display(pd.DataFrame([configuration]))

## Section 7 - YOLO training data without copying the dataset

Ultralytics expects a dataset description. The source WiSARD folders place each `.txt` annotation beside its image instead of using a separate `images/` and `labels/` tree. To avoid modifying or duplicating WiSARD, this notebook creates a project-local **reference view** and a YAML description under `results/rgb`. The references point to the original files; the original files remain where they are.

A training **epoch** is one pass over the training references. A **batch** is a small group processed together before the optimizer updates model weights. Training loss is an internal error signal used to improve predictions. Validation measures behavior on frames not used for weight updates. The test split is not used for tuning.

If the installed Ultralytics version cannot consume this external sibling-label layout directly, the notebook stops with a clear explanation rather than silently training on incorrect labels.

In [ ]:
# Import the standard YOLO-family API. This is intentionally a plain model
# configuration: no FPN changes, attention, transformer, fusion, or alignment.
try:
    from ultralytics import YOLO
    import ultralytics
    print('Ultralytics:', ultralytics.__version__)
except ImportError as error:
    raise ImportError('Install ultralytics in the notebook environment before training: pip install ultralytics') from error

# Ultralytics resolves a sibling annotation by replacing the image suffix
# with .txt for this reference-list layout in supported versions. We write
# only path references and validate that every referenced label exists.
for split_name, frame_ids in split_frames.items():
    reference_file = RGB_RESULTS / f'{split_name}_image_references.txt'
    reference_file.write_text('\n'.join(str(rgb_images[frame].resolve()) for frame in frame_ids) + '\n', encoding='utf-8')
    missing_labels = [frame for frame in frame_ids if frame not in rgb_annotations]
    if missing_labels:
        raise RuntimeError(f'Missing RGB labels for {split_name} frames: {missing_labels}')

DATA_YAML = RGB_RESULTS / 'rgb_wisard_data.yaml'
yaml_text = (
    '# Generated reference-only dataset description.\n'
    f'path: {RGB_RESULTS.as_posix()}\n'
    'train: train_image_references.txt\n'
    'val: val_image_references.txt\n'
    'test: test_image_references.txt\n'
    'nc: 1\n'
    "names: ['class_0']\n"
)
DATA_YAML.write_text(yaml_text, encoding='utf-8')

# Why this patch? The installed Ultralytics version normally writes a label
# cache beside the first source label file. That would violate the project's
# read-only dataset rule. We keep the normal YOLO loader and redirect only its
# cache filename into our project results folder. The image and label paths
# still point to the original dataset; no source file is copied or edited.
from ultralytics.data.dataset import YOLODataset
_original_load_or_scan_cache = YOLODataset._load_or_scan_cache

def load_or_scan_cache_in_project(self, source_cache_path, cache_hash):
    safe_cache_path = RGB_RESULTS / 'ultralytics_label_cache' / source_cache_path.name
    safe_cache_path.parent.mkdir(parents=True, exist_ok=True)
    return _original_load_or_scan_cache(self, safe_cache_path, cache_hash)

YOLODataset._load_or_scan_cache = load_or_scan_cache_in_project
print(DATA_YAML.read_text(encoding='utf-8'))
print('Ultralytics label caches redirected to:', RGB_RESULTS / 'ultralytics_label_cache')

## Section 8 - Train the clean RGB-only baseline

### BEFORE
We use a standard lightweight pretrained YOLO model. Pretraining gives the model general visual features; it does not use WiSARD labels. The baseline remains ordinary RGB detection, with no thermal branch and no architectural modifications.

### DURING
The training call uses the fixed seed, chronological references, aspect-ratio-aware preprocessing, CPU/GPU device selection, and the configuration recorded above. A one-epoch CPU smoke run is the default so the notebook can be validated in a modest environment.

### AFTER
Ultralytics returns a result object and writes run artifacts under `results/rgb`. For a research-quality baseline, increase epochs on a suitable GPU and keep the exact recipe recorded.

### WHY THIS MATTERS
Later thermal and fusion experiments must compare against the same split and a known RGB recipe. Otherwise, an apparent multimodal improvement may be a training-procedure difference.

In [ ]:
# A full run is opt-in because this workspace has no CUDA GPU.
# The default one-epoch run is a reproducibility smoke test, not a final benchmark.
training_epochs = 50 if RUN_FULL_TRAINING else EPOCHS
run_name = 'rgb_baseline_full' if RUN_FULL_TRAINING else 'rgb_baseline_smoke'

model = YOLO(MODEL_NAME)
train_result = model.train(
    data=str(DATA_YAML),
    epochs=training_epochs,
    imgsz=INPUT_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    seed=SEED,
    deterministic=True,
    workers=0,
    cache=False,
    project=str(RGB_RESULTS / 'runs'),
    name=run_name,
    exist_ok=True,
    pretrained=True,
    verbose=True,
)
print('Training finished. Run directory:', RGB_RESULTS / 'runs' / run_name)

### What happened?
The standard RGB detector trained using the fixed chronological references and recorded recipe.

### What does it mean?
A smoke run proves the pipeline can load the source layout, parse labels, resize images, and update a model. It is not enough evidence for a publishable performance claim.

### Is the result reasonable?
Loss should be finite and the run directory should contain training artifacts. A longer GPU run is needed for a meaningful baseline metric.

### What should we do next?
Evaluate validation and test predictions with precision, recall, F1, mAP@50, and mAP@50:95.

## Section 9 - Evaluation metrics

**Precision** asks: when the detector predicts a person, how often is that prediction correct? **Recall** asks: of the annotated people, how many did it find? **F1** is the harmonic mean of precision and recall, so it is high only when both are reasonably high.

**mAP@50** averages precision-recall performance when a predicted box counts as correct at IoU 0.50. IoU is intersection over union: shared box area divided by combined box area. **mAP@50:95** averages over stricter IoU thresholds from 0.50 through 0.95, so it tests localization quality more severely.

Metrics are meaningful only with the split and input resolution recorded. A higher number than a published paper is not automatically a fair win if the datasets, labels, sequence split, image resolution, or evaluation protocol differ. This notebook supports within-project comparisons first.

In [ ]:
# Validation uses the validation block and does not update model weights.
# Test evaluation is reported separately and is not used for tuning.
best_weights = RGB_RESULTS / 'runs' / run_name / 'weights' / 'best.pt'
if not best_weights.exists():
    raise FileNotFoundError(f'Training did not produce expected weights: {best_weights}')

trained_model = YOLO(str(best_weights))
validation_result = trained_model.val(data=str(DATA_YAML), split='val', imgsz=INPUT_SIZE, batch=BATCH_SIZE, device=DEVICE, conf=CONFIDENCE_THRESHOLD, iou=IOU_THRESHOLD, workers=0, cache=False, verbose=True)
test_result = trained_model.val(data=str(DATA_YAML), split='test', imgsz=INPUT_SIZE, batch=BATCH_SIZE, device=DEVICE, conf=CONFIDENCE_THRESHOLD, iou=IOU_THRESHOLD, workers=0, cache=False, verbose=True)

def metric_value(result, name):
    # Ultralytics exposes metrics through a versioned object. This helper
    # keeps extraction readable and records NaN if a field is unavailable.
    try:
        return float(getattr(result.box, name))
    except (AttributeError, TypeError, ValueError):
        return float('nan')

def f1_from_precision_recall(precision, recall):
    # F1 balances precision and recall; avoid division by zero for a
    # smoke run that might make no positive predictions.
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

precision = metric_value(test_result, 'mp')
recall = metric_value(test_result, 'mr')
map50 = metric_value(test_result, 'map50')
map50_95 = metric_value(test_result, 'map')
metrics = pd.DataFrame([{
    'model': MODEL_NAME, 'input_size': INPUT_SIZE,
    'train_images': len(split_frames['train']), 'val_images': len(split_frames['val']), 'test_images': len(split_frames['test']),
    'precision': precision, 'recall': recall, 'f1': f1_from_precision_recall(precision, recall),
    'map50': map50, 'map50_95': map50_95, 'epochs': training_epochs, 'batch_size': BATCH_SIZE, 'seed': SEED,
    'evaluation_note': 'test metrics are descriptive; tune only on validation',
}])
metrics.to_csv(RGB_RESULTS / 'rgb_baseline_metrics.csv', index=False)
display(metrics)

## Section 10 - Qualitative predictions and error analysis

A numeric metric compresses many behaviors into one number. We therefore inspect predictions alongside ground-truth boxes. Green boxes represent annotations; red boxes represent model predictions. We sample validation/test frames from early, middle, late, empty, and crowded categories when possible.

A **successful detection** is a predicted box that overlaps a ground-truth box adequately. A **miss** is an annotated person without a sufficiently overlapping prediction. A **false positive** is a prediction without a matching annotation. We should describe visible patterns as observations, then label explanations as interpretations or hypotheses rather than facts.

The code below uses the trained model's prediction API. If a one-epoch smoke model produces weak predictions, that is an honest pipeline result, not evidence that RGB detection is impossible.

In [ ]:
def draw_ground_truth(axis, image, frame_id):
    # Ground truth is drawn in green so it can be compared with red predictions.
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    for box in yolo_to_pixel_boxes(annotation_data.get(frame_id, []), image.shape):
        axis.add_patch(plt.Rectangle((box['left'], box['top']), box['width'], box['height'], fill=False, edgecolor='lime', linewidth=2, label='ground truth'))

qualitative_frames = split_frames['test'][::max(1, len(split_frames['test']) // 6)][:6]
for frame_id in qualitative_frames:
    prediction = trained_model.predict(source=str(rgb_images[frame_id]), imgsz=INPUT_SIZE, conf=CONFIDENCE_THRESHOLD, iou=IOU_THRESHOLD, device=DEVICE, verbose=False)[0]
    image = cv2.imread(str(rgb_images[frame_id]), cv2.IMREAD_COLOR)
    figure, axis = plt.subplots(figsize=(12, 7))
    draw_ground_truth(axis, image, frame_id)
    if prediction.boxes is not None:
        for coordinates, confidence in zip(prediction.boxes.xyxy.cpu().numpy(), prediction.boxes.conf.cpu().numpy()):
            left, top, right, bottom = coordinates
            axis.add_patch(plt.Rectangle((left, top), right - left, bottom - top, fill=False, edgecolor='red', linewidth=2))
            axis.text(left, top, f'prediction {confidence:.2f}', color='white', backgroundcolor='red')
    axis.set_title(f'Test RGB frame {frame_id}: green=ground truth, red=prediction')
    axis.set_xlabel('pixel x'); axis.set_ylabel('pixel y')
    figure.tight_layout()
    figure.savefig(RGB_FIGURES / f'prediction_frame_{frame_id:04d}.png', dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(figure)

## Section 11 - Save a human-readable baseline summary

This final artifact makes the experiment understandable outside the notebook. It records the dataset split, preprocessing, model settings, metrics, qualitative interpretation, and limitations. For a full GPU run, rerun this cell after training so the summary contains the final metrics.

In [ ]:
summary_text = f'''# RGB Baseline Summary

## Dataset
- Source: `{DATASET_ROOT.resolve()}` (read-only)
- RGB folder: `{RGB_FOLDER.name}`
- Split: chronological contiguous blocks, approximately 70% train / 15% validation / 15% test
- Train images: {len(split_frames['train'])}
- Validation images: {len(split_frames['val'])}
- Test images: {len(split_frames['test'])}
- Native RGB resolution: 3840 x 2160

## Preprocessing
- Model input: {INPUT_SIZE} pixels
- Aspect ratio: preserved with detector letterboxing
- Source images copied: no
- Source images modified: no

## Model and training
- Model: {MODEL_NAME}
- Pretrained: yes
- Epochs: {training_epochs}
- Batch size: {BATCH_SIZE}
- Device: {DEVICE}
- Seed: {SEED}
- Confidence threshold: {CONFIDENCE_THRESHOLD}
- IoU threshold: {IOU_THRESHOLD}

## Test metrics
{metrics.to_string(index=False)}

## Qualitative observations
The prediction figures should be inspected for successful detections, missed small targets, and false positives. The four empty RGB images are useful for checking whether the detector invents people in background-only frames.

## Interpretation
A one-epoch CPU smoke run validates the pipeline but should not be presented as a final performance claim. A longer, fixed GPU run is required for the research baseline. Any later thermal or fusion comparison must reuse the split and document changes explicitly.

## Limitations
- The local annotation files provide class ID 0 but no class-name mapping.
- This is one sequential scene, so chronological test performance may not represent a new location.
- Tiny aerial people may be lost during resizing to the model input.
- Test metrics must not be used for hyperparameter tuning.
'''
(RGB_RESULTS / 'rgb_baseline_summary.md').write_text(summary_text, encoding='utf-8')
print(summary_text)

# Why This Baseline Matters for Our RGB-Thermal Research

## What we learned
The RGB stream has 264 sequential frames, 1,022 annotated boxes, four empty images, and only locally observable class ID 0. The split and preprocessing choices are now explicit and reproducible.

## What worked
The baseline pipeline discovers the external dataset, validates labels, creates reference-only chronological manifests, trains a standard RGB YOLO detector, evaluates it, and saves machine-readable and human-readable results.

## What did not work or remains limited
A CPU smoke run is not a tuned scientific benchmark. The sequence split measures later frames from the same sequence, not a new geographical scene. Class semantics are not supplied by the local files.

## Current RGB baseline metrics
Read `results/rgb/rgb_baseline_metrics.csv`. The exact values depend on whether the notebook was run in smoke mode or with the documented full-training option.

## Main limitations
Small aerial targets, one sequence, no external class-name file, CPU constraints in the current environment, and possible temporal similarity between neighboring frames all limit interpretation.

## What the thermal baseline must reproduce
The thermal baseline must use a deterministic sequence-aware split, preserve its own aspect ratio, record input size and training settings, report the same metrics, and save comparable qualitative/error-analysis artifacts. It must not quietly change the evaluation protocol.

## Why this is the control experiment
If RGB performs strongly, thermal must demonstrate complementary information rather than merely repeating the same signal. If RGB performs poorly, we must separate resolution, small-object difficulty, data scarcity, and model limitations before claiming fusion helps. The future comparisons must begin with this exact RGB condition:

**E0 RGB-only baseline -> E1 thermal-only -> E2 basic fusion -> E3 feature alignment -> E4 attention fusion -> E5 multi-scale fusion.**

No thermal model, fusion model, attention module, feature-alignment module, or architectural modification is implemented in this notebook.

In [ ]:
# Final safety checks: prove that all generated artifacts are outside the
# read-only source dataset and report the source inventory for comparison.
source_files = list(DATASET_ROOT.rglob('*'))
source_file_count = sum(path.is_file() for path in source_files)
generated_files = list(RGB_RESULTS.rglob('*'))
assert all(not path.resolve().is_relative_to(DATASET_ROOT.resolve()) for path in generated_files if path.is_file())
print('Source dataset file count observed:', source_file_count)
print('Generated RGB artifact count:', sum(path.is_file() for path in generated_files))
print('All generated artifacts are outside the source dataset: yes')
print('RGB baseline outputs:', RGB_RESULTS.resolve())